# Notebook 24 — Downtown Code Constraint Analysis

Computes a **composite regulatory constraint score** for each Downtown Code (DTC) character area
and joins KPIs from the existing TOC performance pipeline.

## Outputs
| File | Description |
|---|---|
| `data/dtc/dtc_character_areas_with_kpis.gpkg` | Character area polygons (dissolved from zoning) with constraint scores + KPIs |
| `data/dtc/dtc_parcels_by_character_area.gpkg` | Parcel polygons spatially joined to character areas + constraint scores |
| `outputs/dtc_character_area_kpis.csv` | Tabular KPIs by character area |
| `outputs/dtc_constraint_scores.csv` | Constraint score breakdown per zone code |

## Constraint Score Dimensions (0 = unconstrained, 3 = most constrained)
All values sourced from Phoenix Zoning Ordinance Chapter 12 (Ord. G-7330, 2024).

| Dimension | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| `lot_coverage` | 100% / no max | 75–85% | 55–65% | 50% |
| `height_bonus` | 50%+ available | 25–30% | Conditional / HP-only / sub-area | None |
| `density_bonus` | 100% available | 50% | Conditional / partial | None |
| `parking_reduction` | 100% allowed | 50% | 25% | None |
| `use_permit_burden` | 0 key uses as `up`/`sp`/`np` | 1–2 | 3–5 | 6+ (all key uses restricted) |

**Key uses scored**: Assembly General, Retail Sales, Bar, Restaurant, Hotel/Motel, General Office  
**Scoring**: `np` = 2, `up`/`sp` = 1, `p`/`pc` = 0 per use; summed then bucketed 0–3.

> ⚠️ **Verify after first run**: Constraint scores are derived from the ordinance text and
> should be checked against actual shapefile zone code distributions. The `WEIGHTS` dict
> defaults to equal weights — tune before drawing conclusions.

## 0 · Imports and Paths

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ── File Paths ───────────────────────
REPO_ROOT   = Path("..")
DATA_DIR    = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs" / "dtc"
ZONING_DIR  = DATA_DIR / "zoning"
PARCEL_DIR  = DATA_DIR / "parcels" / "processed"          # assessor parcel shapefile directory
DTC_DIR     = DATA_DIR / "dtc"

DTC_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_CRS = "EPSG:2868"  # NAD83 / Arizona Central (feet)

## 1 · Constraint Score Configuration

All values derived from Phoenix Zoning Ordinance Chapter 12.  
**Edit `WEIGHTS` to tune the composite score before interpreting results.**

In [2]:
# ── Zone code → character area label ──────────────────────────────────────────
# Sourced from user's shapefile ZONING attribute values.
CHARACTER_AREA_MAP: dict[str, str] = {
    "DTC-BIO":    "BioMed",
    "DTC-BCORE":  "Business Core",
    "DTC-CENTP":  "Central Park",
    "DTC-COM-2":  "Commercial Corridors",
    "DTC-COMM1": "Commercial Corridors",
    "DTC-GTWY":   "Downtown Gateway",
    "DTC-EEVER":  "East Evergreen",
    "DTC-E-EV":   "Evans Churchill East",
    "DTC-W-EV":   "Evans Churchill West",
    "DTC-McD-1":  "McDowell Corridor",
    "DTC-McD-2":  "McDowell Corridor",
    "DTC-E-ROO":  "Roosevelt East",
    "DTC-ROOS":   "Roosevelt North",
    "DTC-S-ROO":  "Roosevelt South",
    "DTC-TwnPk":  "Townsend Park",
    "DTC-VANB":   "Van Buren",
    "DTC-WARE":   "Warehouse",
}

# ── Per-dimension constraint scores (0 = unconstrained, 3 = most constrained) ─
#
# Sources (Chapter 12 section numbers):
#   lot_coverage       — §1208–1222 B.3 (Max lot coverage + bonus)
#   height_bonus       — §1208–1222 B.1 (Bonus availability)
#   density_bonus      — §1208–1222 B.2 (Bonus availability)
#   parking_reduction  — §1208–1222 B.5 (Max parking decrease bonus)
#   use_permit_burden  — §1203.C (Land Use Matrix, 6 key uses: Assembly General,
#                        Retail Sales, Bar, Restaurant, Hotel/Motel, General Office)
#
# ⚠️  Evans Churchill East has a split: 50% lot coverage N of Garfield, 90% S.
#     Scored conservatively at 3 (50%) pending spatial sub-area analysis.
# ⚠️  Business Core height bonus is HP-conservation-easement only S of Madison;
#     scored 2 (conditional) rather than 3.
# ⚠️  Downtown Gateway height bonus applies only in sub-area (N of Garfield,
#     S of Roosevelt, E of Central); scored 2 (conditional).

CONSTRAINT_SCORES: dict[str, dict[str, int]] = {
    #                           lot   ht   den  park  up_burden
    "DTC-BIO":    dict(lot_coverage=0, height_bonus=3, density_bonus=1, parking_reduction=0, use_permit_burden=1),
    "DTC-BCORE":  dict(lot_coverage=0, height_bonus=2, density_bonus=3, parking_reduction=0, use_permit_burden=0),
    "DTC-CENTP":  dict(lot_coverage=3, height_bonus=3, density_bonus=1, parking_reduction=1, use_permit_burden=3),
    "DTC-COM-2":  dict(lot_coverage=3, height_bonus=3, density_bonus=3, parking_reduction=1, use_permit_burden=1),
    "DTC-COMM1": dict(lot_coverage=3, height_bonus=3, density_bonus=3, parking_reduction=1, use_permit_burden=1),
    "DTC-GTWY":   dict(lot_coverage=0, height_bonus=2, density_bonus=1, parking_reduction=0, use_permit_burden=1),
    "DTC-EEVER":  dict(lot_coverage=3, height_bonus=3, density_bonus=3, parking_reduction=3, use_permit_burden=1),
    "DTC-E-EV":   dict(lot_coverage=3, height_bonus=2, density_bonus=1, parking_reduction=0, use_permit_burden=1),
    "DTC-W-EV":   dict(lot_coverage=1, height_bonus=0, density_bonus=0, parking_reduction=0, use_permit_burden=1),
    "DTC-McD-1":  dict(lot_coverage=3, height_bonus=3, density_bonus=1, parking_reduction=2, use_permit_burden=1),
    "DTC-McD-2":  dict(lot_coverage=3, height_bonus=3, density_bonus=1, parking_reduction=2, use_permit_burden=1),
    "DTC-E-ROO":  dict(lot_coverage=1, height_bonus=1, density_bonus=0, parking_reduction=0, use_permit_burden=1),
    "DTC-ROOS":   dict(lot_coverage=3, height_bonus=3, density_bonus=0, parking_reduction=1, use_permit_burden=3),
    "DTC-S-ROO":  dict(lot_coverage=3, height_bonus=3, density_bonus=0, parking_reduction=1, use_permit_burden=2),
    "DTC-TwnPk":  dict(lot_coverage=1, height_bonus=1, density_bonus=0, parking_reduction=0, use_permit_burden=1),
    "DTC-VANB":   dict(lot_coverage=2, height_bonus=3, density_bonus=1, parking_reduction=0, use_permit_burden=1),
    "DTC-WARE":   dict(lot_coverage=0, height_bonus=0, density_bonus=0, parking_reduction=0, use_permit_burden=0),
}

# ── Weights (equal to start — tune after reviewing score distribution) ────────
WEIGHTS: dict[str, float] = {
    "lot_coverage":      1.0,
    "height_bonus":      1.0,
    "density_bonus":     1.0,
    "parking_reduction": 1.0,
    "use_permit_burden": 1.0,
}

# ── Constraint threshold: areas above this raw score are flagged ──────────────
# With equal weights, max possible = 15. Default 8.0 flags the upper 40%.
CONSTRAINT_THRESHOLD: float = 8.0

SCORE_DIMS = list(WEIGHTS.keys())

## 2 · Score Computation

In [3]:
def compute_weighted_score(zone_code: str) -> float:
    """Return weighted composite constraint score for a zone code."""
    scores = CONSTRAINT_SCORES[zone_code]
    return sum(scores[dim] * WEIGHTS[dim] for dim in SCORE_DIMS)


# ── Build score reference DataFrame ──────────────────────────────────────────
score_rows = []
for code, dim_scores in CONSTRAINT_SCORES.items():
    row = {"zone_code": code, "character_area": CHARACTER_AREA_MAP[code]}
    row.update({f"score_{k}": v for k, v in dim_scores.items()})
    row["constraint_score"] = compute_weighted_score(code)
    row["is_constrained"] = row["constraint_score"] >= CONSTRAINT_THRESHOLD
    score_rows.append(row)

scores_df = pd.DataFrame(score_rows).sort_values("constraint_score", ascending=False)

# ── Audit detail: ordinance values, raw use-permit counts, notes ──────────────
# One row per character area (scores_df has one row per zone code; two zone codes
# share 'Commercial Corridors', so audit joins on character_area after dedup).

# Key uses evaluated: Assembly General, Retail Sales, Bar, Restaurant,
# Hotel/Motel, General Office  (columns: use, BioMed … Warehouse)
# Values from §1203.C Land Use Matrix: p/pc=0, up/sp=1, np=2
USE_MATRIX_RAW: dict[str, dict[str, int]] = {
    # character_area          : {use: raw_score}
    "BioMed":                  {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Business Core":           {"assembly_general": 0, "retail_sales": 0, "bar": 0, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Central Park":            {"assembly_general": 2, "retail_sales": 2, "bar": 2, "restaurant": 2, "hotel": 2, "general_office": 2},
    "Commercial Corridors":    {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Downtown Gateway":        {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "East Evergreen":          {"assembly_general": 1, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Evans Churchill East":    {"assembly_general": 1, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Evans Churchill West":    {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "McDowell Corridor":       {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Roosevelt East":          {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 1},
    "Roosevelt North":         {"assembly_general": 2, "retail_sales": 2, "bar": 2, "restaurant": 2, "hotel": 2, "general_office": 2},
    "Roosevelt South":         {"assembly_general": 1, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 1, "general_office": 1},
    "Townsend Park":           {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Van Buren":               {"assembly_general": 0, "retail_sales": 0, "bar": 1, "restaurant": 0, "hotel": 0, "general_office": 0},
    "Warehouse":               {"assembly_general": 0, "retail_sales": 0, "bar": 0, "restaurant": 0, "hotel": 0, "general_office": 0},
}

KEY_USES = ["assembly_general", "retail_sales", "bar", "restaurant", "hotel", "general_office"]
USE_LABELS = {"assembly_general": "Assembly General", "retail_sales": "Retail Sales",
              "bar": "Bar", "restaurant": "Restaurant",
              "hotel": "Hotel/Motel", "general_office": "General Office"}
USE_CODE_LABELS = {0: "p/pc", 1: "up/sp", 2: "np"}

audit_detail: dict[str, dict] = {
    # character_area: {ordinance_section, lot_coverage_base, lot_coverage_bonus,
    #                  height_bonus_detail, density_bonus_detail,
    #                  parking_reduction_detail, notes}
    "BioMed":               dict(ordinance_section="§1208", lot_coverage_base="No maximum",    lot_coverage_bonus="None",     height_bonus_detail="None",                                    density_bonus_detail="Max 50%",  parking_reduction_detail="Max 100%",          notes="Biomedical campus; use matrix open but bar is up"),
    "Business Core":        dict(ordinance_section="§1209", lot_coverage_base="No maximum",    lot_coverage_bonus="None",     height_bonus_detail="HP conservation easement only (S of Madison)", density_bonus_detail="None",     parking_reduction_detail="Max 100%",          notes="Most permissive use matrix; density bonus=None is a real constraint"),
    "Central Park":         dict(ordinance_section="§1210", lot_coverage_base="50%",           lot_coverage_bonus="None",     height_bonus_detail="None",                                    density_bonus_detail="Max 50%",  parking_reduction_detail="Max 50%",           notes="Residential scale preservation; all key commercial uses are np"),
    "Commercial Corridors": dict(ordinance_section="§1211", lot_coverage_base="50%",           lot_coverage_bonus="Max 85%",  height_bonus_detail="None",                                    density_bonus_detail="None",     parking_reduction_detail="Max 50%",           notes="Both sub-corridors (7th Ave + Central Ave); density bonus=None"),
    "Downtown Gateway":     dict(ordinance_section="§1212", lot_coverage_base="100%",          lot_coverage_bonus="None",     height_bonus_detail="30% in sub-area only (N of Garfield / S of Roosevelt / E of Central)", density_bonus_detail="Max 50%",  parking_reduction_detail="Max 100%",          notes="High-rise transit corridor; most flexible overall"),
    "East Evergreen":       dict(ordinance_section="§1213", lot_coverage_base="50%",           lot_coverage_bonus="None",     height_bonus_detail="None",                                    density_bonus_detail="None",     parking_reduction_detail="None",              notes="Historic residential; no bonuses of any kind; zero parking reduction allowed"),
    "Evans Churchill East": dict(ordinance_section="§1214", lot_coverage_base="50% (N of Garfield) / 90% (S of Garfield)", lot_coverage_bonus="Max 100%", height_bonus_detail="10% S of Garfield only", density_bonus_detail="Max 50%",  parking_reduction_detail="Max 100%",          notes="Scored conservatively at 50% (N of Garfield); HP buildings present"),
    "Evans Churchill West": dict(ordinance_section="§1215", lot_coverage_base="75%",           lot_coverage_bonus="Max 100%", height_bonus_detail="Up to 100ft additional (Pierce–Fillmore sub-area only)", density_bonus_detail="Max 100%", parking_reduction_detail="Max 100%",          notes="Most flexible non-core area; public market / Roosevelt Row zone"),
    "McDowell Corridor":    dict(ordinance_section="§1216", lot_coverage_base="50%",           lot_coverage_bonus="Max 85%",  height_bonus_detail="None",                                    density_bonus_detail="Max 50%",  parking_reduction_detail="Max 25% only",       notes="Tightest parking flexibility in the code; 5ft side / 15ft rear setbacks"),
    "Roosevelt East":       dict(ordinance_section="§1217", lot_coverage_base="75%",           lot_coverage_bonus="Max 100%", height_bonus_detail="Max 25%",                                 density_bonus_detail="Max 100%", parking_reduction_detail="Max 100%",          notes="Transition zone; moderate constraints; general office and financial institutions are up"),
    "Roosevelt North":      dict(ordinance_section="§1218", lot_coverage_base="50%",           lot_coverage_bonus="Max 75%",  height_bonus_detail="None",                                    density_bonus_detail="Max 100%", parking_reduction_detail="Max 50%",           notes="Historic SFR district; all key commercial uses are np"),
    "Roosevelt South":      dict(ordinance_section="§1219", lot_coverage_base="50%",           lot_coverage_bonus="Max 75%",  height_bonus_detail="None",                                    density_bonus_detail="Max 100%", parking_reduction_detail="Max 50%",           notes="Transitional residential; assembly np; hotel and general office are up"),
    "Townsend Park":        dict(ordinance_section="§1220", lot_coverage_base="75%",           lot_coverage_bonus="Max 100%", height_bonus_detail="Max 30%",                                 density_bonus_detail="Max 100%", parking_reduction_detail="Max 100%",          notes="Mid-rise transition; office/cultural focus; moderate constraints"),
    "Van Buren":            dict(ordinance_section="§1221", lot_coverage_base="55%",           lot_coverage_bonus="Max 100%", height_bonus_detail="None",                                    density_bonus_detail="Max 50%",  parking_reduction_detail="Max 100%",          notes="High-rise intent undermined by no height bonus and unusual 55% base coverage; 10ft interior setback"),
    "Warehouse":            dict(ordinance_section="§1222", lot_coverage_base="100%",          lot_coverage_bonus="None",     height_bonus_detail="Max 50% (or 140ft via HP easement N of Lincoln)", density_bonus_detail="Max 100%", parking_reduction_detail="Max 100% (nonresidential)", notes="Most permissive character area; zero constraints across all dimensions"),
}

# Compute raw use-permit burden per character area
use_rows = []
for area, uses in USE_MATRIX_RAW.items():
    raw_sum = sum(uses.values())
    # Count of uses that are up/sp (1) or np (2)
    restricted_count = sum(1 for v in uses.values() if v > 0)
    np_count = sum(1 for v in uses.values() if v == 2)
    use_detail = "; ".join(
        f"{USE_LABELS[u]}={USE_CODE_LABELS[v]}" for u, v in uses.items() if v > 0
    ) or "all permitted by right"
    use_rows.append({
        "character_area": area,
        "up_burden_raw_sum": raw_sum,
        "key_uses_restricted_count": restricted_count,
        "key_uses_np_count": np_count,
        "use_permit_detail": use_detail,
    })

use_df = pd.DataFrame(use_rows)
audit_df = pd.DataFrame([
    {"character_area": area, **detail}
    for area, detail in audit_detail.items()
])

# ── Assemble the enriched scores CSV ─────────────────────────────────────────
# Deduplicate scores_df to one row per character area before joining audit info
scores_ca = (
    scores_df
    .drop_duplicates("character_area")
    .drop(columns=["zone_code"])
)
enriched_df = (
    scores_ca
    .merge(audit_df, on="character_area", how="left")
    .merge(use_df,   on="character_area", how="left")
    .sort_values("constraint_score", ascending=False)
)

# Also keep a zone-code-level version (scores only, no audit duplication)
scores_zone_level = scores_df.copy()

print(f"Threshold: {CONSTRAINT_THRESHOLD} / max possible {sum(3 * w for w in WEIGHTS.values())}")
print(f"Constrained areas: {enriched_df['is_constrained'].sum()} of {len(enriched_df)}\n")
display(
    enriched_df[
        ["character_area", "constraint_score", "is_constrained"]
        + [f"score_{d}" for d in SCORE_DIMS]
        + ["key_uses_restricted_count", "key_uses_np_count", "use_permit_detail", "notes"]
    ]
)

# Export enriched CSV (one row per character area, all detail columns)
enriched_df.to_csv(OUTPUTS_DIR / "dtc_constraint_scores.csv", index=False)
print("Saved: outputs/dtc_constraint_scores.csv")

Threshold: 8.0 / max possible 15.0
Constrained areas: 6 of 15



,character_area,constraint_score,is_constrained,score_lot_coverage,score_height_bonus,score_density_bonus,score_parking_reduction,score_use_permit_burden,key_uses_restricted_count,key_uses_np_count,use_permit_detail,notes
0,East Evergreen,13.0,True,3,3,3,3,1,2,0,Assembly General=up/sp; Bar=up/sp,Historic residential; no bonuses of any kind; ...
1,Commercial Corridors,11.0,True,3,3,3,1,1,1,0,Bar=up/sp,Both sub-corridors (7th Ave + Central Ave); de...
2,Central Park,11.0,True,3,3,1,1,3,6,6,Assembly General=np; Retail Sales=np; Bar=np; ...,Residential scale preservation; all key commer...
3,McDowell Corridor,10.0,True,3,3,1,2,1,1,0,Bar=up/sp,Tightest parking flexibility in the code; 5ft ...
4,Roosevelt North,10.0,True,3,3,0,1,3,6,6,Assembly General=np; Retail Sales=np; Bar=np; ...,Historic SFR district; all key commercial uses...
5,Roosevelt South,9.0,True,3,3,0,1,2,4,0,Assembly General=up/sp; Bar=up/sp; Hotel/Motel...,Transitional residential; assembly np; hotel a...
6,Van Buren,7.0,False,2,3,1,0,1,1,0,Bar=up/sp,High-rise intent undermined by no height bonus...
7,Evans Churchill East,7.0,False,3,2,1,0,1,2,0,Assembly General=up/sp; Bar=up/sp,Scored conservatively at 50% (N of Garfield); ...
8,BioMed,5.0,False,0,3,1,0,1,1,0,Bar=up/sp,Biomedical campus; use matrix open but bar is up
9,Business Core,5.0,False,0,2,3,0,0,0,0,all permitted by right,Most permissive use matrix; density bonus=None...


Saved: outputs/dtc_constraint_scores.csv


## 3 · Load DTC Zone Polygons

In [4]:
# ── Load generalized zoning shapefile ─────────────────────────────────────────
zoning_raw = gpd.read_file(ZONING_DIR / "Zoning.shp").to_crs(PROJECT_CRS)
print(f"All zoning records: {len(zoning_raw):,}")

# Filter to known DTC zone codes
dtc_zones = zoning_raw[
    zoning_raw["ZONING"].isin(CHARACTER_AREA_MAP.keys())
].copy()

print(f"DTC zone records: {len(dtc_zones):,}")
print("DTC zone code distribution:")
print(dtc_zones["ZONING"].value_counts().to_string())

# Sanity check: flag any codes in shapefile but not in our map
all_dtc_in_file = zoning_raw.loc[
    zoning_raw["ZONING"].str.startswith("DTC-", na=False), "ZONING"
].unique()
unmapped = set(all_dtc_in_file) - set(CHARACTER_AREA_MAP.keys())
if unmapped:
    print(f"\n⚠️  DTC codes in shapefile not in CHARACTER_AREA_MAP: {unmapped}")
    print("   Add these to CHARACTER_AREA_MAP and CONSTRAINT_SCORES before proceeding.")

All zoning records: 9,480
DTC zone records: 102
DTC zone code distribution:
ZONING
DTC-BCORE    23
DTC-WARE     18
DTC-VANB     10
DTC-GTWY      9
DTC-W-EV      7
DTC-E-ROO     6
DTC-S-ROO     6
DTC-BIO       4
DTC-E-EV      3
DTC-McD-1     3
DTC-TwnPk     2
DTC-COM-2     2
DTC-McD-2     2
DTC-EEVER     2
DTC-COMM1     2
DTC-ROOS      2
DTC-CENTP     1


## 4 · Join Constraint Scores to Zone Polygons

In [5]:
dtc_zones = dtc_zones.merge(
    scores_zone_level[["zone_code"] + [f"score_{d}" for d in SCORE_DIMS] + ["constraint_score", "is_constrained", "character_area"]],
    left_on="ZONING",
    right_on="zone_code",
    how="left",
)

print(f"DTC zones with scores: {len(dtc_zones):,}")
print(f"Null scores: {dtc_zones['constraint_score'].isna().sum()} (should be 0)")

DTC zones with scores: 102
Null scores: 0 (should be 0)


## 5 · Dissolve to Character Area Polygons

In [6]:
# Dissolve individual zone polygons into one polygon per character area.
# Score columns are aggregated as the mean (they should be identical within
# an area unless a zone code maps to multiple character areas, which shouldn't
# happen given CHARACTER_AREA_MAP is many-to-one).
score_cols = [f"score_{d}" for d in SCORE_DIMS] + ["constraint_score"]

char_areas = (
    dtc_zones
    .dissolve(
        by="character_area",
        aggfunc={col: "mean" for col in score_cols}
    )
    .reset_index()
)

char_areas["is_constrained"] = char_areas["constraint_score"] >= CONSTRAINT_THRESHOLD
char_areas["area_acres"] = char_areas.geometry.area / 43560.0

print(f"Character areas: {len(char_areas)}")
print(char_areas[["character_area", "constraint_score", "is_constrained", "area_acres"]]
      .sort_values("constraint_score", ascending=False)
      .to_string(index=False))

Character areas: 15
      character_area  constraint_score  is_constrained  area_acres
      East Evergreen              13.0            True   27.595730
Commercial Corridors              11.0            True   43.197754
        Central Park              11.0            True   30.304948
     Roosevelt North              10.0            True  108.364419
   McDowell Corridor              10.0            True   44.894743
     Roosevelt South               9.0            True   48.060387
Evans Churchill East               7.0           False   40.897568
           Van Buren               7.0           False   75.657269
              BioMed               5.0           False   53.047636
       Business Core               5.0           False  368.599568
    Downtown Gateway               4.0           False   82.205310
       Townsend Park               3.0           False   47.754643
      Roosevelt East               3.0           False   33.862493
Evans Churchill West               2.0    

## 6 · KPI Computation per Character Area

Mirrors the KPI pipeline from Notebook 23 but grouped by character area
rather than TOC/Village boundaries.

In [7]:
# ── 6a. Load and clip county parcel GeoPackage to DTC boundary ───────────────
# Fill in your GeoPackage path and layer name, and adjust column name constants
# to match your cleaned assessor dataset.

PARCEL_FILE  = Path(PARCEL_DIR / "maricopa_parcels_with_assessor.gpkg")   # ← absolute or relative path to your parcel gpkg
APN_COL      = "APN"                       # unique parcel identifier
TAXVAL_COL   = "TAXABLE_VA"                # taxable value column
FCV_COL      = "FCV"                       # full cash value column
LAND_VAL_COL = "LAND_VALUE"                # land-only value column

# Build clip mask from the dissolved DTC character area union (already in memory)
dtc_union = char_areas.dissolve().to_crs(PROJECT_CRS)
minx, miny, maxx, maxy = dtc_union.total_bounds

# Pass 1: fast bbox pre-filter via read_file — avoids loading the full county
print("Loading parcel layer within DTC bounding box...")
parcels_raw = gpd.read_file(
    PARCEL_FILE,
    bbox=(minx, miny, maxx, maxy),
).to_crs(PROJECT_CRS)
print(f"  Bbox pass:  {len(parcels_raw):,} parcels")

# Pass 2: overlay to the actual DTC polygon union
parcels_raw = gpd.overlay(parcels_raw, dtc_union).reset_index(drop=True)
print(f"  After overlay: {len(parcels_raw):,} parcels")

Loading parcel layer within DTC bounding box...
  Bbox pass:  4,636 parcels
  After overlay: 2,992 parcels


In [14]:
# ── 6b. Spatial join: parcels → character areas ───────────────────────────────
# Use centroid join to avoid edge slivers inflating counts.
JOIN_COLS_FROM_AREAS = (
    ["character_area", "constraint_score", "is_constrained"]
    + score_cols
)

parcel_centroids = parcels_raw[[APN_COL, "geometry"]].copy()
parcel_centroids["geometry"] = parcel_centroids.geometry.centroid

joined = gpd.sjoin(
    parcel_centroids,
    char_areas[JOIN_COLS_FROM_AREAS + ["geometry"]],
    how="inner",
    predicate="within",
)

# Keep only APN + the columns we brought in from char_areas;
# drop sjoin artifacts (index_right, centroid geometry)
joined = (
    joined
    .drop(columns=[c for c in ["geometry", "index_right"] if c in joined.columns])
    [[APN_COL] + JOIN_COLS_FROM_AREAS]
)

# Drop any columns from parcels_raw that joined is about to supply,
# so pandas can't create _x/_y suffixes on character_area etc.
cols_to_drop = [c for c in JOIN_COLS_FROM_AREAS if c in parcels_raw.columns]
parcels_dtc = parcels_raw.drop(columns=cols_to_drop).merge(joined, on=APN_COL, how="inner")

print(f"Parcels within DTC boundary: {len(parcels_dtc):,}")
print(parcels_dtc.groupby("character_area").size().sort_values(ascending=False))

Parcels within DTC boundary: 2,991
character_area
Business Core           567
Roosevelt South         293
Roosevelt East          291
Warehouse               278
Evans Churchill West    259
Roosevelt North         259
Downtown Gateway        209
Evans Churchill East    189
Van Buren               154
Commercial Corridors    119
Central Park            118
McDowell Corridor        85
East Evergreen           72
Townsend Park            56
BioMed                   42
dtype: int64


In [15]:
# ── 6c. Compute area-weighted KPIs per character area ─────────────────────────

SQ_FT_PER_ACRE = 43560.0

parcels_dtc["parcel_acres"] = parcels_dtc.geometry.area / SQ_FT_PER_ACRE

# Coerce value columns to numeric (assessor data sometimes has nulls/strings)
for col in [TAXVAL_COL, FCV_COL, LAND_VAL_COL]:
    if col in parcels_dtc.columns:
        parcels_dtc[col] = pd.to_numeric(parcels_dtc[col], errors="coerce").fillna(0)


def kpi_by_area(df: pd.DataFrame) -> pd.Series:
    """Aggregate parcel-level data to area-level KPIs."""
    area_acres = df["parcel_acres"].sum()
    result = {
        "parcel_count": len(df),
        "total_acres": area_acres,
    }
    if TAXVAL_COL in df.columns:
        result["total_taxable_value"] = df[TAXVAL_COL].sum()
        result["tvap"] = df[TAXVAL_COL].sum() / area_acres if area_acres > 0 else np.nan
    if FCV_COL in df.columns:
        result["fcv_per_acre"] = df[FCV_COL].sum() / area_acres if area_acres > 0 else np.nan
    if LAND_VAL_COL in df.columns:
        result["land_value_per_acre"] = df[LAND_VAL_COL].sum() / area_acres if area_acres > 0 else np.nan
        # Improvement-to-land value ratio: low ratio = underbuilt/surface lot
        total_land = df[LAND_VAL_COL].sum()
        total_improvement = (df[TAXVAL_COL].sum() - total_land) if TAXVAL_COL in df.columns else np.nan
        result["improvement_to_land_ratio"] = (
            total_improvement / total_land if total_land > 0 else np.nan
        )
    return pd.Series(result)


kpi_df = (
    parcels_dtc
    .groupby("character_area")
    .apply(kpi_by_area, include_groups=False)
    .reset_index()
)

# Merge constraint scores onto KPI table
kpi_df = kpi_df.merge(
    enriched_df[[
        "character_area", "constraint_score", "is_constrained",
        "ordinance_section", "lot_coverage_base", "lot_coverage_bonus",
        "height_bonus_detail", "density_bonus_detail", "parking_reduction_detail",
        "key_uses_restricted_count", "key_uses_np_count", "use_permit_detail", "notes",
    ] + [f"score_{d}" for d in SCORE_DIMS]]
    .drop_duplicates("character_area"),
    on="character_area",
    how="left",
)

kpi_df = kpi_df.sort_values("constraint_score", ascending=False)

display_cols = ["character_area", "constraint_score", "is_constrained", "parcel_count",
                "total_acres", "tvap", "improvement_to_land_ratio"]
display_cols = [c for c in display_cols if c in kpi_df.columns]
display(kpi_df[display_cols].round(2))

,character_area,constraint_score,is_constrained,parcel_count,total_acres
5,East Evergreen,13.0,True,72.0,10.41
3,Commercial Corridors,11.0,True,119.0,25.27
2,Central Park,11.0,True,118.0,20.66
10,Roosevelt North,10.0,True,259.0,60.96
8,McDowell Corridor,10.0,True,85.0,28.53
11,Roosevelt South,9.0,True,293.0,33.34
6,Evans Churchill East,7.0,False,189.0,27.40
13,Van Buren,7.0,False,154.0,53.22
0,BioMed,5.0,False,42.0,36.29
1,Business Core,5.0,False,567.0,284.34


In [20]:
# ── ACS join (block groups → character areas) ─────────────────────────────────
# Reuse the ACS enrichment logic from Notebook 23 if available.
# Expected columns after ACS join: median_income, median_rent, pct_transit_commute

ACS_FILE = DATA_DIR / "census" / "processed" / "tracts_clean.gpkg"   # ← adjust path

tracts_acs = gpd.read_file(
    ACS_FILE
).to_crs(PROJECT_CRS)

# ── ACS field constants (mirroring notebooks 16 / 21) ────────────────────────
TOTAL_POPULATION      = "ASNQE001"
MEDIAN_HH_INCOME      = "ASQPE001"
MEDIAN_RENT           = "ASVBE001"
HOUSING_TOTAL         = "ASS8E001"
OCCUPIED_HH           = "ASS8E002"
VACANT_UNITS          = "ASS8E003"
OWNER_OCCUPIED        = "ASS9E002"
RENTER_OCCUPIED       = "ASS9E003"
TOTAL_WORKERS_16P     = "ASORE001"
DRIVE_ALONE           = "ASORE003"
CARPOOLED             = "ASORE004"
PUBLIC_TRANSPORT      = "ASORE010"
BICYCLE               = "ASORE018"
WALKED                = "ASORE019"
WORKED_FROM_HOME      = "ASORE021"
AUTO_COMMUTERS        = "ASORE002"

ACS_COUNT_COLS = [
    TOTAL_POPULATION, HOUSING_TOTAL, OCCUPIED_HH, VACANT_UNITS,
    OWNER_OCCUPIED, RENTER_OCCUPIED, TOTAL_WORKERS_16P,
    DRIVE_ALONE, CARPOOLED, PUBLIC_TRANSPORT,
    BICYCLE, WALKED, WORKED_FROM_HOME, AUTO_COMMUTERS,
]

for col in ACS_COUNT_COLS + [MEDIAN_HH_INCOME, MEDIAN_RENT]:
    tracts_acs[col] = pd.to_numeric(tracts_acs[col], errors="coerce")

# Mask invalid sentinel values for medians
for col in [MEDIAN_HH_INCOME, MEDIAN_RENT]:
    tracts_acs.loc[(tracts_acs[col] <= 0) | (tracts_acs[col] > 1_000_000), col] = np.nan

tracts_acs["tract_sqft"] = tracts_acs.geometry.area

print("ACS tracts loaded:", len(tracts_acs))

def compute_acs_metrics(tracts_acs, constrained_in_boundary, boundary_id_col):
    """
    Area-weighted ACS interpolation for constrained zones within a boundary.
    Mirrors the approach in notebooks 16 / 21.
    """
    tracts_x_constrained = gpd.overlay(
        tracts_acs,
        constrained_in_boundary[[boundary_id_col, "geometry"]],
        how="intersection",
        keep_geom_type=False
    )
    tracts_x_constrained = tracts_x_constrained[
        tracts_x_constrained.geometry.notna() & ~tracts_x_constrained.geometry.is_empty
    ].copy()

    tracts_x_constrained["intersect_sqft"] = tracts_x_constrained.geometry.area
    tracts_x_constrained["area_weight"]    = (
        tracts_x_constrained["intersect_sqft"] / tracts_x_constrained["tract_sqft"]
    ).clip(upper=1.0)

    # Area-weight all count fields
    for col in ACS_COUNT_COLS:
        tracts_x_constrained[col + "_w"] = (
            tracts_x_constrained[col].fillna(0) * tracts_x_constrained["area_weight"]
        )

    # Household-weighted medians
    tracts_x_constrained["hh_w"]           = (
        tracts_x_constrained[OCCUPIED_HH].fillna(0) * tracts_x_constrained["area_weight"]
    )
    tracts_x_constrained["income_hh_w"]    = (
        tracts_x_constrained[MEDIAN_HH_INCOME].fillna(0)
        * tracts_x_constrained[OCCUPIED_HH].fillna(0)
        * tracts_x_constrained["area_weight"]
    )
    tracts_x_constrained["rent_hh_w"]      = (
        tracts_x_constrained[MEDIAN_RENT].fillna(0)
        * tracts_x_constrained[OCCUPIED_HH].fillna(0)
        * tracts_x_constrained["area_weight"]
    )
    tracts_x_constrained["intersect_acres"] = (
        tracts_x_constrained["intersect_sqft"] / SQ_FT_PER_ACRE
    )

    agg_dict = {"intersect_acres": "sum", "hh_w": "sum",
                "income_hh_w": "sum", "rent_hh_w": "sum"}
    agg_dict.update({col + "_w": "sum" for col in ACS_COUNT_COLS})

    dem = tracts_x_constrained.groupby(boundary_id_col, as_index=False).agg(agg_dict)

    # Rename weighted counts to clean names
    for col in ACS_COUNT_COLS:
        dem[col] = dem[col + "_w"]

    # Approx medians
    dem["median_income_approx"] = np.where(
        dem["hh_w"] > 0, dem["income_hh_w"] / dem["hh_w"], np.nan
    )
    dem["median_rent_approx"] = np.where(
        dem["hh_w"] > 0, dem["rent_hh_w"] / dem["hh_w"], np.nan
    )

    # Density metrics
    dem["pop_per_acre"] = dem[TOTAL_POPULATION] / dem["intersect_acres"]
    dem["hh_per_acre"]  = dem[OCCUPIED_HH] / dem["intersect_acres"]

    # Mode shares (%)
    den = dem[TOTAL_WORKERS_16P].replace(0, np.nan)
    dem["pct_drive_alone"] = (dem[DRIVE_ALONE]        / den) * 100
    dem["pct_transit"]     = (dem[PUBLIC_TRANSPORT]   / den) * 100
    dem["pct_bike"]        = (dem[BICYCLE]            / den) * 100
    dem["pct_walk"]        = (dem[WALKED]             / den) * 100
    dem["pct_wfh"]         = (dem[WORKED_FROM_HOME]   / den) * 100
    dem["pct_auto"]        = (dem[AUTO_COMMUTERS]     / den) * 100

    return dem

if ACS_FILE.exists():
    acs = gpd.read_file(ACS_FILE).to_crs(PROJECT_CRS)
    acs_join = gpd.sjoin(
        acs,
        char_areas[["character_area", "geometry"]],
        how="inner",
        predicate="intersects",
    )
    acs_agg = acs_join.groupby("character_area").agg(
        median_income=("MEDIAN_HH_INCOME", "median"),
        median_rent=("MEDIAN_RENT", "median"),
        pct_transit_commute=("PCT_TRANSIT", "mean"),
    ).reset_index()
    kpi_df = kpi_df.merge(acs_agg, on="character_area", how="left")
    print("ACS data joined.")
else:
    print(f"ACS file not found at {ACS_FILE} — skipping ACS join.")

KeyError: 'ASNQE001'

In [ ]:
# ── LODES join (WAC jobs → character areas via parcel centroids) ───────────────
LODES_FILE = DATA_DIR / "lodes" / "lodes_wac_2022.gpkg"  # ← adjust path

if LODES_FILE.exists():
    lodes = gpd.read_file(LODES_FILE).to_crs(PROJECT_CRS)
    lodes_join = gpd.sjoin(
        lodes,
        char_areas[["character_area", "geometry", "area_acres"]],
        how="inner",
        predicate="within",
    )
    jobs_agg = lodes_join.groupby("character_area").agg(
        total_jobs=("C000", "sum")
    ).reset_index()
    jobs_agg = jobs_agg.merge(
        char_areas[["character_area", "area_acres"]], on="character_area", how="left"
    )
    jobs_agg["jobs_per_acre"] = jobs_agg["total_jobs"] / jobs_agg["area_acres"]
    kpi_df = kpi_df.merge(jobs_agg[["character_area", "total_jobs", "jobs_per_acre"]],
                          on="character_area", how="left")
    print("LODES data joined.")
else:
    print(f"LODES file not found at {LODES_FILE} — skipping LODES join.")

## 7 · Parcel-Level Spatial Layer

Produces `dtc_parcels_by_character_area.gpkg`: individual parcel polygons
with character area label, constraint score, and assessor KPIs attached.
Useful for drill-down cartography and identifying specific underbuilt sites.

In [ ]:
parcel_output_cols = [
    APN_COL,
    "character_area",
    "constraint_score",
    "is_constrained",
    "parcel_acres",
    "geometry",
]

# Add value columns if present
for col in [TAXVAL_COL, FCV_COL, LAND_VAL_COL]:
    if col in parcels_dtc.columns:
        parcel_output_cols.insert(-1, col)

# Compute improvement-to-land ratio at parcel level
if TAXVAL_COL in parcels_dtc.columns and LAND_VAL_COL in parcels_dtc.columns:
    parcels_dtc["improvement_value"] = parcels_dtc[TAXVAL_COL] - parcels_dtc[LAND_VAL_COL]
    parcels_dtc["improvement_to_land_ratio"] = np.where(
        parcels_dtc[LAND_VAL_COL] > 0,
        parcels_dtc["improvement_value"] / parcels_dtc[LAND_VAL_COL],
        np.nan,
    )
    parcel_output_cols.insert(-1, "improvement_value")
    parcel_output_cols.insert(-1, "improvement_to_land_ratio")

parcel_output_cols = [c for c in parcel_output_cols if c in parcels_dtc.columns]
parcels_out = gpd.GeoDataFrame(parcels_dtc[parcel_output_cols], crs=PROJECT_CRS)

print(f"Parcel output layer: {len(parcels_out):,} records")
print(f"Columns: {list(parcels_out.columns)}")

## 8 · Merge KPIs onto Character Area Polygons

In [ ]:
# Attach all KPI columns to the character area GeoDataFrame
char_areas_out = char_areas.merge(
    kpi_df.drop(columns=[f"score_{d}" for d in SCORE_DIMS], errors="ignore"),
    on="character_area",
    how="left",
    suffixes=("", "_kpi"),
)

print(f"Character area output layer: {len(char_areas_out)} records")
print(f"Columns: {list(char_areas_out.columns)}")

## 9 · Export

In [ ]:
# ── GeoPackages ───────────────────────────────────────────────────────────────
char_areas_path = DTC_DIR / "dtc_character_areas_with_kpis.gpkg"
parcels_path    = DTC_DIR / "dtc_parcels_by_character_area.gpkg"

char_areas_out.to_file(char_areas_path, driver="GPKG", layer="character_areas")
print(f"Saved: {char_areas_path}")

parcels_out.to_file(parcels_path, driver="GPKG", layer="parcels")
print(f"Saved: {parcels_path}")

# ── CSVs (no geometry) ────────────────────────────────────────────────────────
kpi_csv_path = OUTPUTS_DIR / "dtc_character_area_kpis.csv"
kpi_df.to_csv(kpi_csv_path, index=False)
print(f"Saved: {kpi_csv_path}")

print("\n✓ All outputs written.")

## 10 · Score Distribution Summary

Quick diagnostic to inform weight tuning.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of composite scores by character area
plot_df = (
    scores_df
    .drop_duplicates("character_area")
    .sort_values("constraint_score", ascending=True)
)
colors = ["#d73027" if v else "#4575b4" for v in plot_df["is_constrained"]]
plot_df.set_index("character_area")["constraint_score"].plot(
    kind="barh", ax=axes[0], color=colors, edgecolor="white"
)
axes[0].axvline(CONSTRAINT_THRESHOLD, color="black", linestyle="--", lw=1, label=f"Threshold ({CONSTRAINT_THRESHOLD})")
axes[0].set_xlabel("Composite Constraint Score")
axes[0].set_title("DTC Character Areas — Constraint Score\n(red = flagged as constrained)")
axes[0].legend()

# Right: heatmap of dimension scores
dim_df = (
    scores_df
    .drop_duplicates("character_area")
    .set_index("character_area")[[f"score_{d}" for d in SCORE_DIMS]]
    .rename(columns={f"score_{d}": d.replace("_", "\n") for d in SCORE_DIMS})
    .sort_values(by=[f"score_{SCORE_DIMS[0]}" for _ in range(1)], ascending=False)
)
im = axes[1].imshow(dim_df.values, cmap="RdYlGn_r", vmin=0, vmax=3, aspect="auto")
axes[1].set_xticks(range(len(dim_df.columns)))
axes[1].set_xticklabels(dim_df.columns, fontsize=8)
axes[1].set_yticks(range(len(dim_df)))
axes[1].set_yticklabels(dim_df.index, fontsize=8)
plt.colorbar(im, ax=axes[1], label="Score (0=unconstrained, 3=most constrained)")
axes[1].set_title("Constraint Dimension Breakdown")

plt.tight_layout()
fig.savefig(OUTPUTS_DIR / "dtc_constraint_score_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/dtc_constraint_score_overview.png")

## Appendix — Score Audit Table

Expand this cell to review the ordinance citations for each score.

In [ ]:
audit = {
    "character_area": [
        "BioMed", "Business Core", "Central Park", "Commercial Corridors",
        "Downtown Gateway", "East Evergreen", "Evans Churchill East",
        "Evans Churchill West", "McDowell Corridor", "Roosevelt East",
        "Roosevelt North", "Roosevelt South", "Townsend Park", "Van Buren", "Warehouse",
    ],
    "ordinance_section": [
        "§1208", "§1209", "§1210", "§1211",
        "§1212", "§1213", "§1214",
        "§1215", "§1216", "§1217",
        "§1218", "§1219", "§1220", "§1221", "§1222",
    ],
    "lot_coverage_base": [
        "No max", "No max", "50%", "50%",
        "100%", "50%", "50% (N of Garfield) / 90% (S)",
        "75%", "50%", "75%",
        "50%", "50%", "75%", "55%", "100%",
    ],
    "lot_coverage_bonus": [
        "None", "None", "None", "Max 85%",
        "None", "None", "Max 100%",
        "Max 100%", "Max 85%", "Max 100%",
        "Max 75%", "Max 75%", "Max 100%", "Max 100%", "None",
    ],
    "height_bonus": [
        "None", "HP easement only S of Madison", "None", "None",
        "30% in sub-area only", "None", "10% S of Garfield only",
        "Up to 100ft (Pierce–Fillmore sub-area)", "None", "Max 25%",
        "None", "None", "Max 30%", "None", "Max 50% (N of Lincoln, 80ft base only)",
    ],
    "density_bonus": [
        "Max 50%", "None", "Max 50%", "None",
        "Max 50%", "None", "Max 50%",
        "Max 100%", "Max 50%", "Max 100%",
        "Max 100%", "Max 100%", "Max 100%", "Max 50%", "Max 100%",
    ],
    "parking_reduction": [
        "Max 100%", "Max 100%", "Max 50%", "Max 50%",
        "Max 100%", "None", "Max 100%",
        "Max 100%", "Max 25%", "Max 100%",
        "Max 50%", "Max 50%", "Max 100%", "Max 100%", "Max 100% (nonres)",
    ],
    "notes": [
        "Biomedical campus zone — use matrix open but bar is up",
        "Most permissive use matrix; density bonus=None is a real constraint",
        "Residential scale preservation; all key commercial uses are np",
        "Both corridors (7th Ave + Central Ave); density bonus=None",
        "High-rise transit corridor; most flexible overall",
        "Historic residential; no bonuses; zero parking reduction",
        "Split north/south character; scored conservatively at 50% base",
        "Most flexible non-core area; 100ft height bonus available",
        "Tightest parking flexibility (25% only); no height/density bonus",
        "Transition zone; moderate constraints; 100% density bonus available",
        "Historic SFR district; all commercial uses prohibited",
        "Transitional residential; assembly np; hotel up; general office up",
        "Mid-rise transition; moderate constraints",
        "High-rise intent but no height bonus; 55% lot coverage is unusual",
        "Most permissive; zero constraints across all dimensions",
    ],
}

audit_df = pd.DataFrame(audit)
display(audit_df)